# Etapa 1: Leitura e Verificação dos Dados


In [0]:
%sql
USE CATALOG PUC_Sprint_2;
USE SCHEMA anp;

In [0]:
display(dbutils.fs.ls("/Volumes/PUC_Sprint_2/anp/raw/bmp"))
display(dbutils.fs.ls("/Volumes/PUC_Sprint_2/anp/raw/bar"))
display(dbutils.fs.ls("/Volumes/PUC_Sprint_2/anp/raw/bdep"))
display(dbutils.fs.ls("/Volumes/PUC_Sprint_2/anp/raw/eco"))

In [0]:
df_bmp = spark.read.csv("/Volumes/PUC_Sprint_2/anp/raw/bmp/*.csv", header=True, inferSchema=True)
df_bar = spark.read.csv("/Volumes/PUC_Sprint_2/anp/raw/bar/*.csv", header=True, inferSchema=True)
df_bdep = spark.read.csv("/Volumes/PUC_Sprint_2/anp/raw/bdep/*.csv", header=True, inferSchema=True)
df_eco = spark.read.csv("/Volumes/PUC_Sprint_2/anp/raw/eco/*.csv", header=True, inferSchema=True)

OBS 1- correção da leitura errada do ; no csp do BDEP + correção do ISO para evitar símbolos

In [0]:
df_bdep = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .csv("/Volumes/PUC_Sprint_2/anp/raw/bdep/*.csv"))

df_bar = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ",")
    .option("encoding", "ISO-8859-1")
    .csv("/Volumes/PUC_Sprint_2/anp/raw/bar/*.csv"))

df_bdmp = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .csv("/Volumes/PUC_Sprint_2/anp/raw/bmp/*.csv"))

df_cambio = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/PUC_Sprint_2/anp/raw/economico/cambio_usd_brl.csv"))

df_brent = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/PUC_Sprint_2/anp/raw/economico/brent_spot.csv"))

df_cambio.printSchema()
df_brent.printSchema()
df_bar.printSchema()
df_bdep.printSchema()
df_bmp.printSchema()


In [0]:
import re

def sanitizar_para_delta(df):
    """O mínimo necessário para o Delta aceitar a tabela: troca só os
    caracteres proibidos por '_'. Mantém acento, maiúscula, tudo o resto
    como veio no arquivo original - isso é trabalho da Silver, não daqui."""
    novos_nomes = [re.sub(r"[ ,;{}()\n\t=/³]+", "_", c).strip("_") for c in df.columns]
    return df.toDF(*novos_nomes)

In [0]:
df_bmp_bronze = sanitizar_para_delta(df_bmp)
df_bar_bronze = sanitizar_para_delta(df_bar)
df_bdep_bronze = sanitizar_para_delta(df_bdep)
df_brent_bronze = sanitizar_para_delta(df_brent)
df_cambio_bronze = sanitizar_para_delta(df_cambio)

# Criação das tabelas

In [0]:
df_bmp_bronze.write.mode("overwrite").saveAsTable("PUC_Sprint_2.anp.bronze_bmp")
df_bar_bronze.write.mode("overwrite").saveAsTable("PUC_Sprint_2.anp.bronze_bar")
df_bdep_bronze.write.mode("overwrite").saveAsTable("PUC_Sprint_2.anp.bronze_bdep")
df_cambio_bronze.write.mode("overwrite").saveAsTable("PUC_Sprint_2.anp.bronze_cambio")
df_brent_bronze.write.mode("overwrite").saveAsTable("PUC_Sprint_2.anp.bronze_brent")

In [0]:
for tabela in ["bronze_bmp", "bronze_bar", "bronze_bdep","bronze_cambio","bronze_brent"]:
    existe = spark.catalog.tableExists(f"PUC_Sprint_2.anp.{tabela}")
    print(f"{tabela}: {'existe' if existe else 'AINDA NÃO EXISTE'}")